# 🔵 K-Means Clustering – Furniture Shop ML Service

Notebook này trình bày phương pháp **K-Means Clustering** được triển khai trong `ml_service/app/services/clustering_service.py`.

## Tổng quan – Hai loại phân cụm:

| Loại | Đặc trưng | Nhãn cụm có thể có |
|------|-----------|---------------------|
| **Product Clustering** | price, rating, sold, stock, img_count, has_3d | Premium, Bán chạy, Tồn kho cao, Giá tốt... |
| **User Clustering** | total_orders, total_spent, avg_order, reviews | VIP, Mua thường xuyên, Giá trị cao, Phổ thông |

## 1. Thiết lập môi trường

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from collections import defaultdict

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('✅ Thư viện đã sẵn sàng')

## 2. Nạp dữ liệu

In [ ]:
products_raw = pd.read_csv('furniture_shop.products.csv')
reviews_raw  = pd.read_csv('furniture_shop.reviews.csv')
orders_raw   = pd.read_csv('furniture_shop.orders.csv')
categories   = pd.read_csv('furniture_shop.categories.csv')
brands       = pd.read_csv('furniture_shop.brands.csv')

cat_map   = dict(zip(categories['_id'].astype(str), categories['name']))
brand_map = dict(zip(brands['_id'].astype(str), brands['name']))
print(f'📦 Products: {len(products_raw)}, ⭐ Reviews: {len(reviews_raw)}, 🛒 Orders: {len(orders_raw)}')

## 3. PART A – Phân cụm Sản phẩm (Product Clustering)

### Các đặc trưng sử dụng:
```
feature_cols = ['price', 'average_rating', 'total_reviews',
                'sold_count', 'stock', 'image_count', 'has_3d']
```

In [ ]:
# BUILD PRODUCT FEATURES
prod_rows = []
for _, p in products_raw.iterrows():
    img_cols  = [c for c in products_raw.columns if c.startswith('images[')]
    img_count = int(pd.notna(p[img_cols]).sum())
    has_3d    = 1.0 if (pd.notna(p.get('model3DUrl','')) and str(p.get('model3DUrl','')) not in ('', 'nan')) else 0.0

    prod_rows.append({
        'product_id'    : str(p['_id']),
        'name'          : str(p.get('name', '')),
        'category'      : cat_map.get(str(p.get('category','')), ''),
        'brand'         : brand_map.get(str(p.get('brand','')), ''),
        'price'         : float(p.get('price', 0) or 0),
        'average_rating': float(p.get('averageRating', 0) or 0),
        'total_reviews' : float(p.get('totalReviews', 0) or 0),
        'sold_count'    : float(p.get('soldCount', 0) or 0),
        'stock'         : float(p.get('stock', 0) or 0),
        'image_count'   : float(img_count),
        'has_3d'        : has_3d,
    })

prod_df = pd.DataFrame(prod_rows)
print(f'✅ Product feature DataFrame: {prod_df.shape}')

# Xem phân phối của các features
feature_cols_p = ['price','average_rating','total_reviews','sold_count','stock','image_count','has_3d']
prod_df[feature_cols_p].describe()

In [ ]:
# Visualise feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(feature_cols_p):
    axes[i].hist(prod_df[col], bins=min(10, prod_df[col].nunique()), color='#3498db', edgecolor='white')
    axes[i].set_title(col, fontweight='bold', fontsize=9)
    axes[i].set_ylabel('Count')

axes[-1].axis('off')
plt.suptitle('Phân phối các Feature của Sản phẩm (trước chuẩn hoá)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.1 StandardScaler → K-Means

In [ ]:
N_CLUSTERS_PRODUCTS = 4

X_prod   = prod_df[feature_cols_p].values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_prod)

# K-Means với n_init=20 (như trong clustering_service.py)
kmeans_p = KMeans(n_clusters=N_CLUSTERS_PRODUCTS, random_state=42, n_init=20)
prod_df['cluster'] = kmeans_p.fit_predict(X_scaled)

print(f'✅ Phân cụm xong: {N_CLUSTERS_PRODUCTS} cụm sản phẩm')
print('\nPhân phối sản phẩm theo cụm:')
print(prod_df['cluster'].value_counts().sort_index())

### 3.2 Rule-Based Cluster Labeling

Mỗi cụm được gán nhãn dựa trên so sánh với ngưỡng quantile của toàn bộ dataset.

In [ ]:
# PRODUCT_LABEL_RULES từ cluster_label_config.py
PRODUCT_LABEL_RULES = [
    {'label': 'Premium chất lượng cao', 'all': [
        ('price', '>=', 'price_p75'), ('average_rating', '>=', 'rating_p75')]},
    {'label': 'Bán chạy', 'all': [
        ('sold_count', '>=', 'sold_p75'), ('total_reviews', '>=', 'reviews_p75')]},
    {'label': 'Tồn kho cao bán chậm', 'all': [
        ('stock', '>=', 'stock_p75'), ('sold_count', '<=', 'sold_p25')]},
    {'label': 'Giá tốt bán ổn', 'all': [
        ('price', '<=', 'price_p25'), ('sold_count', '>=', 'sold_p50')]},
]
PRODUCT_DEFAULT_LABEL = 'Nhóm phổ thông'

def compute_thresholds(df):
    return {
        'price_p25'   : df['price'].quantile(0.25),
        'price_p75'   : df['price'].quantile(0.75),
        'rating_p75'  : df['average_rating'].quantile(0.75),
        'sold_p25'    : df['sold_count'].quantile(0.25),
        'sold_p50'    : df['sold_count'].quantile(0.50),
        'sold_p75'    : df['sold_count'].quantile(0.75),
        'reviews_p75' : df['total_reviews'].quantile(0.75),
        'stock_p75'   : df['stock'].quantile(0.75),
    }

def compare(left, op, right):
    ops = {'>=': left>=right, '<=': left<=right, '>': left>right, '<': left<right, '==': left==right}
    return ops.get(op, False)

def pick_label(cluster_stats, thresholds, rules, default):
    for rule in rules:
        all_match = all(
            compare(float(cluster_stats.get(f, 0)), op, float(thresholds.get(t, 0)))
            for f, op, t in rule['all']
        )
        if all_match:
            return rule['label']
    return default

thresholds = compute_thresholds(prod_df)
print('📐 Ngưỡng phân loại (quantile):')
for k, v in thresholds.items():
    print(f'   {k:<15}: {v:,.1f}')

In [ ]:
# Tính cluster stats và gán nhãn
cluster_summaries = {}
for cid in sorted(prod_df['cluster'].unique()):
    rows = prod_df[prod_df['cluster'] == cid]
    means = {col: float(rows[col].mean()) for col in feature_cols_p}
    label = pick_label(means, thresholds, PRODUCT_LABEL_RULES, PRODUCT_DEFAULT_LABEL)
    cluster_summaries[cid] = {'cluster': cid, 'label': label, 'size': len(rows), 'avg_features': means}

prod_df['cluster_label'] = prod_df['cluster'].map({k: v['label'] for k, v in cluster_summaries.items()})

# Hiển thị tóm tắt
print('\n📊 CLUSTER SUMMARIES – PRODUCTS')
print('='*60)
for cid, s in cluster_summaries.items():
    print(f"\nCụm {cid}: [{s['label']}] – {s['size']} sản phẩm")
    for feat, val in s['avg_features'].items():
        print(f"    {feat:<18}: {val:>10,.2f}")

In [ ]:
# Visualise kết quả Product Clustering
CLUSTER_COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
unique_clusters = sorted(prod_df['cluster'].unique())
color_map = {c: CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i, c in enumerate(unique_clusters)}

# PCA để visualise 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
prod_df['pca1'] = X_pca[:, 0]
prod_df['pca2'] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PCA scatter
for cid in unique_clusters:
    mask = prod_df['cluster'] == cid
    label = cluster_summaries[cid]['label']
    axes[0].scatter(prod_df[mask]['pca1'], prod_df[mask]['pca2'],
                    c=color_map[cid], label=f'Cụm {cid}: {label}', s=100, alpha=0.85, edgecolors='white', linewidths=1)
    # Annotate product names
    for _, row in prod_df[mask].iterrows():
        axes[0].annotate(row['name'][:12], (row['pca1'], row['pca2']),
                         textcoords='offset points', xytext=(4, 4), fontsize=6, alpha=0.8)

axes[0].set_title(f'Product Clusters – PCA 2D\n(Explained: {pca.explained_variance_ratio_.sum()*100:.1f}%)', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(fontsize=7, loc='best')

# Bar chart cluster sizes
labels = [f"Cụm {k}\n{v['label'][:20]}" for k, v in cluster_summaries.items()]
sizes  = [v['size'] for v in cluster_summaries.values()]
colors = [color_map[k] for k in cluster_summaries.keys()]
bars   = axes[1].bar(labels, sizes, color=colors, edgecolor='white', linewidth=1.5)
axes[1].bar_label(bars, fmt='%d SP', padding=3, fontsize=10)
axes[1].set_title('Số Sản Phẩm trong mỗi Cụm', fontweight='bold')
axes[1].set_ylabel('Số sản phẩm')
axes[1].tick_params(axis='x', labelsize=8)

plt.suptitle('K-Means Product Clustering (k=4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('product_clustering_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: product_clustering_result.png')

In [ ]:
# Radar chart – Feature profile of each cluster
from matplotlib.patches import FancyArrowPatch

features_display = feature_cols_p
n_feat = len(features_display)
angles = np.linspace(0, 2*np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

# Normalize cluster means to [0,1]
cluster_means = pd.DataFrame({k: v['avg_features'] for k, v in cluster_summaries.items()}).T
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min() + 1e-9)

fig, axes = plt.subplots(1, len(cluster_summaries), figsize=(16, 4), subplot_kw=dict(polar=True))
if len(cluster_summaries) == 1:
    axes = [axes]

for i, (cid, ax) in enumerate(zip(sorted(cluster_summaries.keys()), axes)):
    values = cluster_means_norm.loc[cid, features_display].tolist()
    values += values[:1]
    color = CLUSTER_COLORS[i % len(CLUSTER_COLORS)]

    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(features_display, fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25','0.5','0.75'], fontsize=6)
    ax.set_title(f"Cụm {cid}\n{cluster_summaries[cid]['label']}", fontsize=8, fontweight='bold', pad=10)

plt.suptitle('Radar Chart – Feature Profile của từng Cụm Sản phẩm\n(Giá trị đã chuẩn hoá)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. PART B – Phân cụm Người dùng (User Clustering)

### Các đặc trưng sử dụng:
```
feature_cols = ['total_orders', 'total_qty', 'total_spent',
                'avg_order_value', 'total_reviews', 'avg_rating']
```
Tất cả tính từ lịch sử Orders + Reviews.

In [ ]:
# BUILD USER FEATURES (tái hiện kmeans_clustering() cluster_type='users')
user_stats = defaultdict(lambda: {'total_orders':0.0,'total_qty':0.0,'total_spent':0.0,
                                   'total_reviews':0.0,'avg_rating_sum':0.0})

item_p_cols = [c for c in orders_raw.columns if c.startswith('items[') and c.endswith('.product')]
item_q_cols = [c for c in orders_raw.columns if c.startswith('items[') and c.endswith('.quantity')]

for _, o in orders_raw.iterrows():
    uid = str(o['user'])
    user_stats[uid]['total_orders'] += 1
    user_stats[uid]['total_spent']  += float(o.get('totalAmount', 0) or 0)
    for qc in item_q_cols:
        qty = o.get(qc)
        if pd.notna(qty):
            user_stats[uid]['total_qty'] += float(qty)

for _, r in reviews_raw.iterrows():
    uid = str(r['user'])
    user_stats[uid]['total_reviews']  += 1
    user_stats[uid]['avg_rating_sum'] += float(r.get('rating', 0) or 0)

user_rows = []
for uid, stats in user_stats.items():
    avg_rating = stats['avg_rating_sum'] / stats['total_reviews'] if stats['total_reviews'] else 0
    avg_order  = stats['total_spent']   / stats['total_orders']   if stats['total_orders']  else 0
    user_rows.append({
        'user_id'       : uid,
        'total_orders'  : stats['total_orders'],
        'total_qty'     : stats['total_qty'],
        'total_spent'   : stats['total_spent'],
        'avg_order_value': avg_order,
        'total_reviews' : stats['total_reviews'],
        'avg_rating'    : avg_rating,
    })

user_df = pd.DataFrame(user_rows)
feature_cols_u = ['total_orders','total_qty','total_spent','avg_order_value','total_reviews','avg_rating']
print(f'✅ User feature DataFrame: {user_df.shape}')
user_df.describe()

In [ ]:
N_CLUSTERS_USERS = 4
n_users = len(user_df)
n_clusters_u = max(2, min(N_CLUSTERS_USERS, n_users)) if n_users > 1 else 1

X_user   = user_df[feature_cols_u].values
scaler_u = StandardScaler()
X_usscaled = scaler_u.fit_transform(X_user)

kmeans_u = KMeans(n_clusters=n_clusters_u, random_state=42, n_init=20)
user_df['cluster'] = kmeans_u.fit_predict(X_usscaled)

print(f'✅ Phân cụm xong: {n_clusters_u} cụm người dùng')
print(user_df['cluster'].value_counts().sort_index())

In [ ]:
# USER LABEL RULES từ cluster_label_config.py
USER_LABEL_RULES = [
    {'label': 'Khách VIP trung thành', 'all': [
        ('total_spent', '>=', 'spent_p75'), ('total_orders', '>=', 'orders_p75')]},
    {'label': 'Mua thường xuyên', 'all': [
        ('total_orders', '>=', 'orders_p75'), ('avg_order_value', '<=', 'avg_order_p50')]},
    {'label': 'Giá trị đơn cao', 'all': [
        ('avg_order_value', '>=', 'avg_order_p75'), ('total_orders', '<=', 'orders_p50')]},
    {'label': 'Tương tác cao', 'all': [
        ('total_reviews', '>=', 'reviews_p75')]},
]
USER_DEFAULT_LABEL = 'Khách hàng phổ thông'

u_thresh = {
    'spent_p75'     : user_df['total_spent'].quantile(0.75),
    'orders_p50'    : user_df['total_orders'].quantile(0.50),
    'orders_p75'    : user_df['total_orders'].quantile(0.75),
    'avg_order_p50' : user_df['avg_order_value'].quantile(0.50),
    'avg_order_p75' : user_df['avg_order_value'].quantile(0.75),
    'reviews_p75'   : user_df['total_reviews'].quantile(0.75),
}

user_summaries = {}
for cid in sorted(user_df['cluster'].unique()):
    rows  = user_df[user_df['cluster'] == cid]
    means = {col: float(rows[col].mean()) for col in feature_cols_u}
    label = pick_label(means, u_thresh, USER_LABEL_RULES, USER_DEFAULT_LABEL)
    user_summaries[cid] = {'cluster': cid, 'label': label, 'size': len(rows), 'avg_features': means}

user_df['cluster_label'] = user_df['cluster'].map({k: v['label'] for k, v in user_summaries.items()})

print('\n📊 CLUSTER SUMMARIES – USERS')
print('='*60)
for cid, s in user_summaries.items():
    print(f"Cụm {cid}: [{s['label']}] – {s['size']} users")
    for feat, val in s['avg_features'].items():
        print(f"    {feat:<20}: {val:>12,.2f}")

In [ ]:
# Visualise User Clustering
unique_clusters_u = sorted(user_df['cluster'].unique())
color_map_u = {c: CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i, c in enumerate(unique_clusters_u)}

if len(X_usscaled[0]) > 2:
    pca_u = PCA(n_components=2, random_state=42)
    X_pca_u = pca_u.fit_transform(X_usscaled)
    user_df['pca1'] = X_pca_u[:, 0]
    user_df['pca2'] = X_pca_u[:, 1]
    expl = pca_u.explained_variance_ratio_.sum() * 100
else:
    user_df['pca1'] = X_usscaled[:, 0]
    user_df['pca2'] = X_usscaled[:, 1]
    expl = 100.0

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for cid in unique_clusters_u:
    mask  = user_df['cluster'] == cid
    label = user_summaries[cid]['label']
    axes[0].scatter(user_df[mask]['pca1'], user_df[mask]['pca2'],
                    c=color_map_u[cid], label=f'Cụm {cid}: {label}', s=120,
                    alpha=0.85, edgecolors='white', linewidths=1.5)
    for _, row in user_df[mask].iterrows():
        axes[0].annotate(f"...{row['user_id'][-6:]}", (row['pca1'], row['pca2']),
                         textcoords='offset points', xytext=(4, 4), fontsize=6, alpha=0.75)

axes[0].set_title(f'User Clusters – PCA 2D (Explained: {expl:.1f}%)', fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8)

# Stacked bars: features per cluster
cluster_feature_df = pd.DataFrame({
    f"Cụm {k}\n{v['label'][:18]}": v['avg_features'] for k, v in user_summaries.items()
}).T

cluster_feature_df[['total_orders', 'total_reviews']].plot(kind='bar', ax=axes[1],
    color=['#3498db','#f39c12'], edgecolor='white', linewidth=0.5)
axes[1].set_title('Avg Orders & Reviews per Cluster', fontweight='bold')
axes[1].set_ylabel('Avg Count')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(fontsize=9)

plt.suptitle('K-Means User Clustering (k=4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('user_clustering_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: user_clustering_result.png')

In [ ]:
# Spending analysis per cluster
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cluster_labels_list = [f"C{k}:{v['label'][:14]}" for k, v in user_summaries.items()]

# Total spent
for cid in unique_clusters_u:
    rows = user_df[user_df['cluster'] == cid]
    axes[0].bar(f"C{cid}", rows['total_spent'].mean() / 1e6,
                color=color_map_u[cid], edgecolor='white')
axes[0].set_title('Avg Total Spent per Cluster\n(triệu VND)', fontweight='bold')
axes[0].set_ylabel('Triệu VND')

# Avg order value
for cid in unique_clusters_u:
    rows = user_df[user_df['cluster'] == cid]
    axes[1].bar(f"C{cid}", rows['avg_order_value'].mean() / 1e6,
                color=color_map_u[cid], edgecolor='white')
axes[1].set_title('Avg Order Value per Cluster\n(triệu VND)', fontweight='bold')
axes[1].set_ylabel('Triệu VND')

# Total orders
for cid in unique_clusters_u:
    rows = user_df[user_df['cluster'] == cid]
    axes[2].bar(f"C{cid}", rows['total_orders'].mean(),
                color=color_map_u[cid], edgecolor='white')
axes[2].set_title('Avg Total Orders per Cluster', fontweight='bold')
axes[2].set_ylabel('Số đơn hàng')

for ax in axes:
    ax.set_xlabel('Cluster')

# Common legend
patches = [mpatches.Patch(color=color_map_u[c], label=f"C{c}: {user_summaries[c]['label']}")
           for c in unique_clusters_u]
fig.legend(handles=patches, loc='lower center', ncol=len(unique_clusters_u),
           bbox_to_anchor=(0.5, -0.12), fontsize=8)

plt.suptitle('Phân tích Hành vi Mua sắm theo Nhóm User', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Elbow Method – Tìm k tối ưu

In [ ]:
# Elbow method cho Product clustering
inertia_p = []
K_range   = range(2, min(8, len(prod_df)))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_p.append(km.inertia_)

inertia_u = []
K_range_u = range(2, min(8, len(user_df)))
for k in K_range_u:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_usscaled)
    inertia_u.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_range), inertia_p, marker='o', color='#3498db', linewidth=2, markersize=8)
axes[0].axvline(x=N_CLUSTERS_PRODUCTS, color='red', linestyle='--', alpha=0.7, label=f'k={N_CLUSTERS_PRODUCTS} (đang dùng)')
axes[0].set_title('Elbow Method – Product Clustering', fontweight='bold')
axes[0].set_xlabel('Số cụm k')
axes[0].set_ylabel('Inertia (SSE)')
axes[0].legend()

axes[1].plot(list(K_range_u), inertia_u, marker='s', color='#2ecc71', linewidth=2, markersize=8)
axes[1].axvline(x=n_clusters_u, color='red', linestyle='--', alpha=0.7, label=f'k={n_clusters_u} (đang dùng)')
axes[1].set_title('Elbow Method – User Clustering', fontweight='bold')
axes[1].set_xlabel('Số cụm k')
axes[1].set_ylabel('Inertia (SSE)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Kết quả cuối – Mỗi item trong cụm của nó

In [ ]:
print('📦 PRODUCT CLUSTERING RESULTS:')
for cid in sorted(prod_df['cluster'].unique()):
    members = prod_df[prod_df['cluster'] == cid]
    label   = cluster_summaries[cid]['label']
    print(f'\n  Cụm {cid} – [{label}] ({len(members)} SP):')
    for _, row in members.iterrows():
        print(f'    • {row["name"]:<35} price={row["price"]/1e6:.1f}M | sold={row["sold_count"]:.0f} | rating={row["average_rating"]:.1f}')

In [ ]:
print('\n👥 USER CLUSTERING RESULTS:')
for cid in sorted(user_df['cluster'].unique()):
    members = user_df[user_df['cluster'] == cid]
    label   = user_summaries[cid]['label']
    print(f'\n  Cụm {cid} – [{label}] ({len(members)} users):')
    for _, row in members.iterrows():
        print(f'    • ...{row["user_id"][-12:]:<15} orders={row["total_orders"]:.0f} | "
              f"spent={row["total_spent"]/1e6:.1f}M | avg_order={row["avg_order_value"]/1e6:.1f}M | "
              f"reviews={row["total_reviews"]:.0f}')

## 7. Tóm tắt luồng xử lý

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         K-Means Clustering – Luồng Xử Lý                        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  PRODUCT CLUSTERING:                                             ║
║  1. products_to_df() → DataFrame với features số                 ║
║  2. features: price, avg_rating, total_reviews,                  ║
║               sold_count, stock, image_count, has_3d             ║
║  3. StandardScaler → chuẩn hoá                                   ║
║  4. KMeans(n_clusters=k, n_init=20) → cluster labels            ║
║  5. Rule-based labeling dựa trên quantile thresholds            ║
║                                                                  ║
║  USER CLUSTERING:                                                ║
║  1. Tổng hợp từ orders: total_orders, total_spent, total_qty     ║
║  2. Tổng hợp từ reviews: total_reviews, avg_rating               ║
║  3. → avg_order_value = total_spent / total_orders               ║
║  4. StandardScaler → KMeans → Rule-based labeling               ║
║                                                                  ║
║  OUTPUT:                                                         ║
║  {cluster_type, cluster_count, cluster_summaries, clusters}      ║
╚══════════════════════════════════════════════════════════════════╝
""")